# Walbot MCP — agent workflow test

Walbot runs in two steps: **step 1** proposes a portfolio from market data (no brokerage),
**step 2** turns a reviewed proposal into orders (no market data). The scheduled runner
chains them directly; the agent's only addition is validation in between.

Start the server first:

```
python -m src.mcp_server --host 127.0.0.1 --port 8001
```

Step 3 below (validating the proposal) is done by the agent with its own web-search tool,
so it is only sketched here.

In [1]:
import json
from contextlib import asynccontextmanager

from mcp.client.session import ClientSession
from mcp.client.sse import sse_client

SERVER_URL = "http://127.0.0.1:8001/sse"
ALGORITHM = "fast_momentum"


@asynccontextmanager
async def connect():
    """Open a short-lived MCP session.

    One session per cell: anyio ties a session's cancel scope to the task that opened it,
    and each notebook cell runs in its own task, so a session held across cells fails on close.
    """
    async with sse_client(SERVER_URL) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            yield session


async def call(name, **arguments):
    """Call an MCP tool and return its parsed JSON payload."""
    async with connect() as session:
        result = await session.call_tool(name, arguments=arguments)

    text = result.content[0].text if result.content else ""
    if result.isError:
        # Server-side exceptions come back as plain text, not JSON, so surface them as-is.
        raise RuntimeError(f"{name} failed: {text}")
    return json.loads(text) if text else None


print("Configured for", SERVER_URL)

Configured for http://127.0.0.1:8001/sse


In [2]:
async with connect() as session:
    tools = await session.list_tools()

for tool in tools.tools:
    params = list(tool.inputSchema.get("properties", {}))
    print(f"{tool.name}({', '.join(params)})")

get_algorithm_result(algorithm)
get_current_positions()
place_orders(algorithm_result, target_weights)


## 1. Algorithm result — step 1, market data only

In [3]:
result = await call("get_algorithm_result", algorithm=ALGORITHM)

print(f"strategy   : {result['strategy']}")
print(f"as_of      : {result['as_of']}")
print(f"allocation : {result['allocation_mode']}")
print(f"priced     : {len(result['latest_prices'])} symbols\n")

print("proposed target weights:")
for symbol, weight in sorted(result["target_weights"].items(), key=lambda kv: -kv[1]):
    row = result["signals"].get(symbol, {})
    score = row.get("score")
    print(f"  {symbol:<6} {weight:>7.2%}  score={round(score, 3) if score is not None else '-':<8} {row.get('reason', '-')}")

strategy   : fast_momentum
as_of      : 2026-08-09T01:46:26.510507+00:00
allocation : Dynamic rank
priced     : 16 symbols

proposed target weights:
  BBC     44.01%  score=1.595    Top Rank
  GTEK    26.79%  score=0.971    Top Rank
  GLD     12.65%  score=0.459    Top Rank
  XSD     11.55%  score=0.418    Top Rank


In [4]:
# Step 1 needs no brokerage at all — this is the whole payload step 2 will consume.
print("top scores across the universe (selected or not):")
ranked = sorted(result["signals"].items(), key=lambda kv: -(kv[1].get("score") or 0))
for symbol, row in ranked[:8]:
    held = "selected" if result["target_weights"].get(symbol) else ""
    print(f"  {symbol:<6} score={row.get('score'):>7.3f}  {row.get('reason', '-'):<14} {held}")

top scores across the universe (selected or not):
  BBC    score=  1.595  Top Rank       selected
  GTEK   score=  0.971  Top Rank       selected
  GLD    score=  0.459  Top Rank       selected
  XSD    score=  0.418  Top Rank       selected
  GVIP   score=  0.329  No rank slot   
  AIQ    score=  0.103  Micro too low  
  QQQM   score= -0.108  Score too low  
  IEMG   score= -0.207  Score too low  


## 2. Current positions — brokerage source of truth

In [5]:
positions = await call("get_current_positions")

print(f"account     : {positions['account_id']}")
print(f"equity      : ${positions['equity']:,.2f}")
print(f"cash        : ${positions['cash']:,.2f}")
print(f"buying power: ${positions['buying_power']:,.2f}\n")

for row in positions["positions"]:
    price = result["latest_prices"].get(row["symbol"])
    value = f"${row['shares'] * price:,.2f}" if price else "n/a"
    print(f"  {row['symbol']:<6} {row['shares']:>10} shares  {value:>14}")
if not positions["positions"]:
    print("  (flat)")

account     : paper
equity      : $9,319.28
cash        : $-20.45
buying power: $24,560.55

  BBC            71 shares       $3,728.78
  GTEK           61 shares       $3,561.74
  IEMG           12 shares         $960.12
  XSD             2 shares       $1,090.30


## 3. Validate the proposed changes

In production the agent researches each add/drop with its own web-search tool. That is
deliberately outside the trading bot's MCP surface, so here we just note what it would check.

In [6]:
held = {row["symbol"] for row in positions["positions"]}
proposed = set(result["target_weights"])

for symbol in sorted(proposed - held):
    print(f"ADD  {symbol:<6} -> research before accepting")
for symbol in sorted(held - proposed):
    print(f"DROP {symbol:<6} -> confirm the exit is intended")
for symbol in sorted(held & proposed):
    print(f"KEEP {symbol:<6} -> resize only")

ADD  GLD    -> research before accepting
DROP IEMG   -> confirm the exit is intended
KEEP BBC    -> resize only
KEEP GTEK   -> resize only
KEEP XSD    -> resize only


## 4. Place orders with the reviewed set — step 2, brokerage only

Pass the step-1 payload back unchanged; it carries the prices used to size shares. Supply
`target_weights` **only** to override the proposal — it is then the complete intended
portfolio, so anything held but not listed is sold to zero.

This submits real orders to the paper account.

In [7]:
# Submit the algorithm's own proposal: pass no override.
override = None

# Example override — drop a name research rejected and hold the cash instead:
# override = {s: w for s, w in result["target_weights"].items() if s != "GTEK"}

if override is None:
    print("submitting the algorithm's proposal unchanged")
else:
    total = sum(override.values())
    print(f"overriding with {len(override)} positions, {total:.2%} of equity")
    assert total <= 1.0, "weights must sum to <= 1.0"

submitting the algorithm's proposal unchanged


In [8]:
placed = await call("place_orders", algorithm_result=result, target_weights=override)

print("status:", placed["status"])
for order in placed.get("order_results", []):
    if order.get("status") == "rejected":
        print(f"  REJECTED {order['action']:<4} {order['symbol']:<6} qty={order['quantity']:<12} {order['reason']}")
    else:
        print(f"  {order['action']:<4} {order['symbol']:<6} qty={order['quantity']:<12} order_id={order.get('order_id','-')}")
if not placed.get("order_results"):
    print("reason:", placed.get("reason"))

status: partial
  sell GTEK   qty=19.17        order_id=d7ca411c-fa90-418c-b54b-e30e830047f9
  REJECTED sell IEMG   qty=12           {"available":"0","code":40310000,"existing_qty":"12","held_for_orders":"12","message":"insufficient qty available for order (requested: 12, available: 0)","related_orders":["c3573277-86ea-4ae8-871e-70fc4a39ea01"],"symbol":"IEMG"}
  buy  BBC    qty=5.41         order_id=4861f1bf-164e-42eb-b826-4e3774393377
  buy  GLD    qty=2.89         order_id=5bc325fc-724c-4086-9f05-f10e540d9ca5


### The weight diff — what step 2 actually changed

In [9]:
print(f"{'symbol':<8}{'current':>10}{'final':>10}{'change':>10}  action")
for row in placed["diff"]:
    print(f"{row['symbol']:<8}{row['current_weight']:>9.2%}{row['final_weight']:>10.2%}{row['change']:>+10.2%}  {row['action']}")

symbol     current     final    change  action
GLD         0.00%    12.63%   +12.63%  add
GTEK       38.22%    26.75%   -11.47%  trim
IEMG       10.30%     0.00%   -10.30%  trim
BBC        40.01%    43.94%    +3.93%  add
XSD        11.70%    11.68%    -0.02%  trim


### Validation errors are returned, not raised

In [10]:
print(await call("place_orders", algorithm_result=result, target_weights={"SPY": 0.7, "QQQ": 0.5}))
print(await call("place_orders", algorithm_result=result, target_weights={}))

# A symbol step 1 never priced cannot be introduced at step 2.
print(await call("place_orders", algorithm_result=result, target_weights={"NVDA": 0.5}))


{'strategy': 'fast_momentum', 'status': 'error', 'reason': 'Target weights sum to 1.2000, which exceeds 1.0 (100% of equity)'}
{'strategy': 'fast_momentum', 'status': 'error', 'reason': 'target_weights must be a non-empty mapping of symbol to weight'}


{'strategy': 'fast_momentum', 'status': 'error', 'reason': 'No price available for NVDA. Step 2 does not fetch market data, so a symbol must have been priced by the algorithm run it came from.'}


## 5. Confirm the resulting positions

Share counts only change once the orders fill. Submitting outside market hours queues them
for the next open, so this cell will still show the pre-trade holdings until then.

In [11]:
after = await call("get_current_positions")
for row in after["positions"]:
    print(f"  {row['symbol']:<6} {row['shares']:>10} shares")
print(f"\ncash: ${after['cash']:,.2f}")

  BBC            71 shares
  GTEK           61 shares
  IEMG           12 shares
  XSD             2 shares

cash: $-20.45
